<a href="https://colab.research.google.com/github/Poojithasingh07/machinelearning/blob/main/Song%20Recommendation%20System%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [4]:
df = pd.read_csv("songs.csv", engine='python', on_bad_lines='warn')

/tmp/ipython-input-3713762849.py:1: ParserWarning: Skipping line 56008: unexpected end of data

  df = pd.read_csv("songs.csv", engine='python', on_bad_lines='warn')


In [5]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [6]:
df.tail(5)

,artist,song,link,text
56001,Westlife,Lost In You,/w/westlife/lost+in+you_10186824.html,There's no more waiting \r\nHolding out for l...
56002,Westlife,Love Can Build A Bridge,/w/westlife/love+can+build+a+bridge_20363383.html,I'd gladly walk across the desert \r\nWith no...
56003,Westlife,Moon River,/w/westlife/moon+river_20260276.html,Moon River wider than a mile \r\nI'm crossing...
56004,Westlife,More Than Words,/w/westlife/more+than+words_20145953.html,Saying I love you \r\nIs not the words \r\nI...
56005,Westlife,My Girl,/w/westlife/my+girl_20267455.html,I've got sunshine on a cloudy day \r\nWhen it...


In [7]:
df.shape

(56006, 4)

In [8]:
df.isnull().sum()

,0
artist,0
song,0
link,0
text,0


In [9]:
df =df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [10]:
df.head(10)

,artist,song,text
0,John Mellencamp,I'm Not Running Anymore,"Holly told me, ""You better give me a child"" \..."
1,Kris Kristofferson,From The Bottle To The Bottom,You ask me if I'm happy now \r\nThat's good a...
2,Whitney Houston,Heartbreak Hotel,This is the heart break hotel \r\nThis is the...
3,Ray Charles,How Long Has This Been Going On,I could cry salty tears \r\nWhere have I been...
4,Loretta Lynn,Secret Love,Once I had a secret love that lived within the...
5,Great Big Sea,Old Brown's Daughter,There is an ancient party at the other end of ...
6,Warren Zevon,Dirty Little Religion,"Warren Zevon, Zevon Music BMI \r\nI like to t..."
7,Ufo,Treacle People,"I walked through this place, although it wasn'..."
8,Ne-Yo,So Sick,"Gotta change my answering machine, now that I'..."
9,System Of A Down,X,"Tell the people, \r\nTell the people that arr..."


In [11]:
df['text'][0]

'Holly told me, "You better give me a child"  \r\nI said, "Holly, there\'s no way  \r\nWe don\'t even like each other all that much  \r\nWe couldn\'t make it one more day"  \r\nShe said, "You better look out, buster  \r\nThe next time you see me you\'re gonna pay"  \r\nI said, "Holly, I\'m not running anymore  \r\nBut I\'m on my way"  \r\n  \r\nI\'m on my way  \r\nI\'m on my way  \r\nI\'m on my way  \r\nBut I\'m not running anymore  \r\n  \r\nWell I got two circus clowns here who like to fight  \r\nThey got one black eye and a bloody nose  \r\nThey are the hoodlums of my third wife  \r\nWhatever I say they will oppose  \r\nI try to teach those clowns something  \r\nLike how to make it day to day  \r\nI say, "Hey, you kids, I\'m not running anymore  \r\nBut I\'m on my way"  \r\n  \r\nI\'m on my way  \r\nI\'m on my way  \r\nI\'m on my way  \r\nAnd I\'m not running anymore  \r\n  \r\nWell I look in the mirror - what the hell happened to me?  \r\nWhatever I had has gone away  \r\nI\'m not 

In [12]:
df.shape

(5000, 3)

In [13]:
df['text'] = df['text'].str.lower().replace(r'^\w\s', ' ').replace(r'\n', ' ', regex = True)

In [18]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Adding download for punkt_tab as specifically requested by the error
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [19]:
df['text'] = df['text'].apply(lambda x: tokenization(x))

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)

In [22]:
similarity[0]

array([1.        , 0.05522968, 0.03193342, ..., 0.05365613, 0.02466558,
       0.01734947])

In [23]:
df[df['song'] == 'Crying Over You']

,artist,song,text
616,UB40,Crying Over You,cri over you in the morn cri over you in the e...


In [24]:
def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similarity[idx])),reverse=True,key=lambda x:x[1])

    songs = []
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)

    return songs

In [25]:
recommendation('Crying Over You')

['Cry Baby Cry',
 'Crying',
 'A Sentimental Blues',
 'Hope For Love',
 'Cry Me A River',
 'My Heart Cries For You',
 "Big Boys Don't Cry",
 "Don't Cry",
 'Fly Away',
 'Cry For Love',
 "Don't Cry Baby",
 'Too Late To Cry',
 'Sad Movies',
 "I Won't Cry For You",
 'Handsome Hands',
 'A Lovely Place To Cry',
 'M-E',
 'Nobody Knows The Way I Feel This Morning',
 "Don't Come Cryin' To Me",
 'Seven Lonely Days']

In [26]:
import pickle
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))

In [28]:
!pip install streamlit
import streamlit as st
import pickle

st.set_page_config(page_title="Music Recommender", layout="centered")

@st.cache_resource
def load_data():
    df = pickle.load(open("df.pkl", "rb"))
    similarity = pickle.load(open("similarity.pkl", "rb"))
    return df, similarity

df, similarity = load_data()

SONG_COL = "song"  # change if needed

st.title("🎵 Music Recommendation System")

song_selected = st.selectbox(
    "Choose a song",
    sorted(df[SONG_COL].dropna().unique())
)

def recommend(song_name):
    idx = df[df[SONG_COL] == song_name].index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    return df.iloc[[i[0] for i in scores]][SONG_COL]

if st.button("Recommend"):
    st.subheader("Recommended Songs")
    for song in recommend(song_selected):
        st.write("🎶", song)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 120.2 MB/s eta 0:00:00


2025-12-30 07:59:04.002 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 07:59:04.004 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 07:59:04.090 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-30 07:59:04.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 07:59:04.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 07:59:04.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-30 07:59:04.242 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [29]:
!ls


df.pkl	sample_data  similarity.pkl  songs.csv


In [30]:
%%writefile app.py
import streamlit as st
import pickle

st.set_page_config(page_title="Music Recommender", layout="centered")

@st.cache_resource
def load_data():
    df = pickle.load(open("df.pkl", "rb"))
    similarity = pickle.load(open("similarity.pkl", "rb"))
    return df, similarity

df, similarity = load_data()

SONG_COL = "song"  # change ONLY if your column name is different

st.title("🎵 Music Recommendation System")
st.write("Select a song to get recommendations")

song_selected = st.selectbox(
    "Choose a song",
    sorted(df[SONG_COL].dropna().unique())
)

def recommend(song_name):
    idx = df[df[SONG_COL] == song_name].index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    return df.iloc[[i[0] for i in scores]][SONG_COL]

if st.button("Recommend"):
    st.subheader("Recommended Songs")
    for song in recommend(song_selected):
        st.write("🎶", song)


Writing app.py


In [31]:
!ls


app.py	df.pkl	sample_data  similarity.pkl  songs.csv


In [32]:
mkdir ~/music-recommender
cd ~/music-recommender


SyntaxError: invalid syntax (ipython-input-1395095215.py, line 1)

In [33]:
from google.colab import files
files.download("app.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
from google.colab import files
files.download("app.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>